# Pipeline architecture figure — reproducibility notebook

Regenerates `figures/fig_methods_pipeline.png` from `provenance/pipeline_metrics.json`.

**Inputs:** `../../provenance/pipeline_metrics.json` (relative to this notebook in `figures/notebooks/`).
**Output:** `../fig_methods_pipeline.png`.

In the template's initial state (no pipeline run yet), all 20 phases render as
`scheduled` (faded outline). Phase 13 of a real run rewrites
`pipeline_metrics.json` with executed-phase data and re-executes this notebook;
Phase 20a re-executes once more after the final phases complete, flipping every
phase's status to `complete` with its outcome annotation drawn from the gate
artifacts.

All values plotted on the figure are sourced from the JSON; no values are
hard-coded in this notebook.

In [ ]:
import json
import pathlib

METRICS_PATH = pathlib.Path('../../provenance/pipeline_metrics.json')
OUT_PATH     = pathlib.Path('../fig_methods_pipeline.png')

with open(METRICS_PATH) as f:
    metrics = json.load(f)

print('pipeline_version:', metrics.get('pipeline_version', '(unset)'))
print('title           :', metrics.get('title', '(unset)'))
print('phases tracked  :', len(metrics.get('phases', [])))

In [ ]:
def render_pipeline_figure(metrics, outpath):
    """Render the 20-phase pipeline diagram.

    metrics["phases"] is a list of dicts:
        {"phase": int, "label": str, "role": "coord"|"agent"|"critic",
         "status": "complete"|"in_progress"|"scheduled",
         "annotation": str}

    Missing or empty `metrics["phases"]` -> render a default 20-phase scaffold
    in `scheduled` state with no annotations.
    """
    import matplotlib.pyplot as plt
    from matplotlib.patches import FancyBboxPatch

    DEFAULT_PHASES = [
        ( 1, "Scope",                "coord"),
        ( 2, "Evidence\nGathering",  "agent"),
        ( 3, "Citation\nInfra",      "coord"),
        ( 4, "Evidence\nCompliance", "coord"),
        ( 5, "Evidence\nCuration",   "agent"),
        ( 6, "Scaffold",             "coord"),
        ( 7, "Section\nDrafting",    "agent"),
        ( 8, "Section\nCritics",     "critic"),
        ( 9, "Bibliography",         "coord"),
        (10, "Integration",          "coord"),
        (11, "Intro /\nConclusion",  "agent"),
        (12, "Bookend\nCritic",      "critic"),
        (13, "Methods",              "agent"),
        (14, "Document\nAssembly",   "coord"),
        (15, "Citation\nTriples",    "critic"),
        (16, "Verification",         "coord"),
        (17, "Fix\nPreparation",     "coord"),
        (18, "Fix\nExecution",       "coord"),
        (19, "Fix\nApplication",     "coord"),
        (20, "Repository\nPush",     "coord"),
    ]

    phases_data = metrics.get("phases") or []
    phase_lookup = {p["phase"]: p for p in phases_data}

    phases = []
    for n, label, role in DEFAULT_PHASES:
        p = phase_lookup.get(n, {})
        phases.append({
            "n":          n,
            "label":      p.get("label", label),
            "role":       p.get("role", role),
            "status":     p.get("status", "scheduled"),
            "annotation": p.get("annotation", ""),
        })

    EDGE = {"coord": "#2e7d32", "agent": "#e65100", "critic": "#6a1b9a"}
    FILL = {"coord": "#c8e6c9", "agent": "#ffe0b2", "critic": "#e1bee7"}
    EDGE_W = {"complete": 2.2, "in_progress": 2.8, "scheduled": 1.2}
    ALPHA  = {"complete": 1.0, "in_progress": 1.0, "scheduled": 0.45}
    LINESTYLE = {"complete": "solid", "in_progress": "solid", "scheduled": "dashed"}

    fig, ax = plt.subplots(figsize=(16, 7))
    ax.set_xlim(0, 21); ax.set_ylim(-2.5, 4.5)
    ax.set_aspect("equal"); ax.axis("off")

    box_w, box_h = 0.9, 1.4
    for p in phases:
        x = p["n"]
        ax.add_patch(FancyBboxPatch(
            (x - box_w/2, 0), box_w, box_h,
            boxstyle="round,pad=0.05",
            facecolor=FILL[p["role"]], edgecolor=EDGE[p["role"]],
            linewidth=EDGE_W[p["status"]], linestyle=LINESTYLE[p["status"]],
            alpha=ALPHA[p["status"]],
        ))
        ax.text(x, box_h/2, p["label"],
                ha="center", va="center", fontsize=8,
                alpha=ALPHA[p["status"]])
        ax.text(x, box_h + 0.18, str(p["n"]),
                ha="center", va="bottom", fontsize=9, weight="bold",
                alpha=ALPHA[p["status"]])
        if p["annotation"]:
            ax.text(x, -0.25, p["annotation"],
                    ha="center", va="top", fontsize=7,
                    color="#555", style="italic")

    for i in range(1, 20):
        ax.annotate("",
            xy=(i + 1 - box_w/2, box_h/2),
            xytext=(i + box_w/2, box_h/2),
            arrowprops=dict(arrowstyle="->", lw=0.8, color="#888"))

    legend_handles = [
        plt.Line2D([0], [0], marker="s", linestyle="", markersize=12,
                   markerfacecolor=FILL["coord"], markeredgecolor=EDGE["coord"],
                   label="Coordinator (mechanical)"),
        plt.Line2D([0], [0], marker="s", linestyle="", markersize=12,
                   markerfacecolor=FILL["agent"], markeredgecolor=EDGE["agent"],
                   label="EXPERT / DATAML actor"),
        plt.Line2D([0], [0], marker="s", linestyle="", markersize=12,
                   markerfacecolor=FILL["critic"], markeredgecolor=EDGE["critic"],
                   label="Blinded critic"),
    ]
    ax.legend(handles=legend_handles, loc="lower center",
              bbox_to_anchor=(0.5, -0.15), ncol=3, frameon=False, fontsize=9)

    title = metrics.get("title", "Computational Review")
    ax.set_title(f"Expert Review Pipeline v25 - {title}",
                 fontsize=11, weight="bold", pad=15)

    plt.tight_layout()
    fig.savefig(outpath, dpi=300, bbox_inches="tight", facecolor="white")
    return fig

In [ ]:
fig = render_pipeline_figure(metrics, str(OUT_PATH))
print('saved:', OUT_PATH)

## Notes

- **Box colors** encode role: green = coordinator-owned mechanical phase, orange = EXPERT or DATAML actor delegation, purple = blinded critic phase.
- **Box outlines** encode status: solid + bold = complete, solid + extra-bold = in-progress, dashed + faded = scheduled (not yet executed).
- **Numeric annotations** below each phase are drawn from the corresponding `phases[i].annotation` string in `pipeline_metrics.json`. Update the JSON and re-run this notebook to refresh the figure.
- The template ships an empty `phases: []` so all phases render as `scheduled`. Phase 13 of a real pipeline run rewrites `phases` with executed entries; Phase 20a (final ledger refresh) re-executes this notebook after Phases 14-20 complete. The validator check `METHODS_LEDGER_FRESH` (sub-check 4) asserts no `scheduled`/`in_progress` tuples remain after Phase 20a.